In [1]:
import pandas as pd
from src.pipelines.BERT_pipeline import BERTPipeline
from src.pipelines.SBERT_pipeline import SBERTPipeline
import logging
import torch
import os

In [2]:
df = pd.read_csv("data/aes_dataset_5k_clean.csv")
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [3]:
# Check if the first file exists
df_result = None
if os.path.exists("experiments/results/results_model_siamese.csv"):
    df_result = pd.read_csv("experiments/results/results_model_siamese.csv")
    print(df_result['config_id'].iloc[-1])
else:
    print("File 'results_model_siamese.csv' does not exist.")

20


In [4]:
batch_sizes = [16]
learning_rates = [5e-5, 1e-4] # , 
warm_ups = [0.0, 0.3]
idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
ROOT_DIR = os.getcwd()

In [5]:
for batch_size in batch_sizes:
    for lr in learning_rates:
        for warm_up in warm_ups:
            results = []
            results_epoch = []
            df_result1 = None
            # Check if the second file exists
            if os.path.exists("experiments/results/results_epoch_siamese.csv"):
                df_result1 = pd.read_csv("experiments/results/results_epoch_siamese.csv")
                print(max(df_result1['valid_pearson']))
            else:
                print("File 'results_epoch_siamese.csv' does not exist.")

            # set up hyperparamter
            config = {
                "df": df,
                # "model_name": "indobenchmark/indobert-lite-base-p2",
                "model_name": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
                # "model_name": "all-MiniLM-L6-v2",
                "batch_size": batch_size,
                "learning_rate": lr,
                "epochs": 100,
                "config_id": idx,
                "best_valid_pearson": max(df_result1['valid_pearson']) if df_result1 is not None and not df_result1.empty else float("-inf"),
                "warmup_ratio": warm_up
            }

            logging.info(
                f"Running configuration: config_id={idx}, model_name={config['model_name']}"
                f", batch_size={batch_size}, epochs={100}, learning_rate={lr}"
            )
            
            print(
                f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}"
                f", batch_size={batch_size}, epochs={100}, learning_rate={lr}"
            )
            
            try:
                pipeline = SBERTPipeline(config, results, results_epoch)
                pipeline.training()

                # Save results
                # Dapatkan root project
                results_path = os.path.join(ROOT_DIR, "experiments/results/results_model_siamese.csv")
                results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch_siamese.csv")
                BERTPipeline.save_csv(results, results_path)
                BERTPipeline.save_csv(results_epoch, results_epoch_path)
            except Exception as e:
                logging.error(f"Error in config_id={idx}: {str(e)}")
                print(f"Error in config_id={idx}: {str(e)}")
                torch.cuda.empty_cache()
            finally:
                # Clear GPU memory after every configuration
                del pipeline.model
                del pipeline.optimizer
                torch.cuda.empty_cache()

            idx += 1

0.9293318951398708

Running configuration: config_id=21, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=16, epochs=100, learning_rate=5e-05
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======


c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\xlm_roberta\modeling_xlm_roberta.py:371: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/100 - Avg training loss: 0.0248, MAE: 0.1174, RMSE: 0.1575, Pearson Corr: 0.8407
Avg validation loss: 0.0206, MAE: 0.1091, RMSE: 0.1406, Pearson Corr: 0.8584
Validation loss decreased (inf --> 0.020644). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0109, MAE: 0.07983, RMSE: 0.1045, Pearson Corr: 0.9279
Avg validation loss: 0.0175, MAE: 0.1004, RMSE: 0.1323, Pearson Corr: 0.879
Validation loss decreased (0.020644 --> 0.017543). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0077, MAE: 0.06776, RMSE: 0.08762, Pearson Corr: 0.949
Avg validation loss: 0.0178, MAE: 0.09815, RMSE: 0.1289, Pearson Corr: 0.882
EarlyStopping counter: 1 out of 10
====== Training Epoch 4/100 ======
Epoch 4/100 - Avg training loss: 0.0056, MAE: 0.05657, RMSE: 0.07481, Pearson Corr: 0.9625
Avg validation loss: 0.0129, MAE: 0.08418, RMSE: 0.1133, Pearson Corr: 0.9048
Validation loss decreased (0.017543 --> 0.012946). Saving mod

c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:120: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt

Avg testing loss: 0.0110, MAE: 0.07736, RMSE: 0.1052, Pearson Corr: 0.9168
0.9293318951398708

Running configuration: config_id=22, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=16, epochs=100, learning_rate=5e-05
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======
Epoch 1/100 - Avg training loss: 0.0897, MAE: 0.2513, RMSE: 0.2995, Pearson Corr: 0.7079
Avg validation loss: 0.0730, MAE: 0.2016, RMSE: 0.2556, Pearson Corr: 0.6847
Validation loss decreased (inf --> 0.072984). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0286, MAE: 0.1307, RMSE: 0.1691, Pearson Corr: 0.819
Avg validation loss: 0.0183, MAE: 0.1034, RMSE: 0.1333, Pearson Corr: 0.8703
Validation loss decreased (0.072984 --> 0.018312). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0133, MAE: 0.08938, RMSE: 0.1151, Pearson Corr: 0.9069
Avg

...

Avg testing loss: 0.0116, MAE: 0.08225, RMSE: 0.108, Pearson Corr: 0.9118
0.9293318951398708

Running configuration: config_id=23, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=16, epochs=100, learning_rate=0.0001
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======
Epoch 1/100 - Avg training loss: 0.0299, MAE: 0.1327, RMSE: 0.1729, Pearson Corr: 0.8133
Avg validation loss: 0.0267, MAE: 0.1269, RMSE: 0.1617, Pearson Corr: 0.8114
Validation loss decreased (inf --> 0.026672). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0178, MAE: 0.1026, RMSE: 0.1334, Pearson Corr: 0.8859
Avg validation loss: 0.0193, MAE: 0.1035, RMSE: 0.1398, Pearson Corr: 0.8688
Validation loss decreased (0.026672 --> 0.019283). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0118, MAE: 0.08287, RMSE: 0.1085, Pearson Corr: 0.9221
Av

c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:120: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt

Avg testing loss: 0.0108, MAE: 0.07564, RMSE: 0.1042, Pearson Corr: 0.9187
0.9293318951398708

Running configuration: config_id=24, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=16, epochs=100, learning_rate=0.0001
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======
Epoch 1/100 - Avg training loss: 0.0747, MAE: 0.2223, RMSE: 0.2734, Pearson Corr: 0.705
Avg validation loss: 0.0401, MAE: 0.146, RMSE: 0.1896, Pearson Corr: 0.7763
Validation loss decreased (inf --> 0.040109). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0177, MAE: 0.1023, RMSE: 0.1331, Pearson Corr: 0.8772
Avg validation loss: 0.0138, MAE: 0.08979, RMSE: 0.1171, Pearson Corr: 0.8974
Validation loss decreased (0.040109 --> 0.013759). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0100, MAE: 0.07786, RMSE: 0.1002, Pearson Corr: 0.9296
Av

c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:120: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt

Avg testing loss: 0.0107, MAE: 0.07885, RMSE: 0.1037, Pearson Corr: 0.9218
